Python notebook to create/parse the datasets used for backbone swap training (Pile-NER, CC related datasets, and CliReNER).

Borrows from [preprocess_other_datasets.ipynb](https://github.com/P0L3/CliReNER/blob/main/preprocess_other_ner_datasets.ipynb).

# Data and Libraries

In [ ]:
# !pip install catalogue
# !pip install confection

In [2]:
import os
import json
import random
import argparse
from collections import defaultdict
 
from datasets import load_dataset
# import spacy
 
from dataset_processing import (
    IBMCCNER_DIR, IBMCCNER_LABELS, ibmccner_process_bio_documents,
    BIODIVNER_DIR, BIODIVNER_LABELS, biodivner_process_bio_documents,
    char_spans_to_gliner2_examples
)
 


## Functions

In [3]:
def load_ibmccner_char_spans(dataset_id=IBMCCNER_DIR, split=("train", "validation", "test")):
    print(f"[IBMCCNER] Loading '{dataset_id}' splits={split}")
    ds = load_dataset(dataset_id)
    train_documentwise = []
    temp_list = []
    for sp in split:
        for line in ds[sp]["text"]:
            if line.strip().startswith("-DOCSTART-"):
                if temp_list:
                    train_documentwise.append(temp_list)
                temp_list = []
            temp_list.append(line)
        if temp_list:
            train_documentwise.append(temp_list)
            temp_list = []
 
    print("[IBMCCNER] Converting BIO -> char spans")
    structured = ibmccner_process_bio_documents(
        document_list=train_documentwise,
        labels_to_keep=IBMCCNER_LABELS,
    )
    print(f"[IBMCCNER] {len(structured)} sentences")
    return structured
 
 
def load_biodivner_char_spans(data_dir=BIODIVNER_DIR, split=("train", "test", "dev")):
    print(f"[BioDivNER] Loading '{data_dir}' splits={split}")
    structured = biodivner_process_bio_documents(
        data_dir=data_dir,
        labels_to_keep=BIODIVNER_LABELS,
        split=list(split),
    )
    print(f"[BioDivNER] {len(structured)} sentences")
    return structured
 
 
def load_climateie_char_spans(data_dir="DATA/ClimateIE/human_corpus/", spacy_model="en_core_sci_sm"):
    """
    ClimateIE ships as whole-document JSONs with a flat `entities` dict
    keyed by opaque IDs; character offsets are GLOBAL (document-level).
    We sentence-split with spaCy and remap entity offsets to be
    sentence-local, keeping only entities fully contained in one sentence.
    Mirrors SCRIPT_VERSIONS/preprocess_other_ner_datasets.py exactly.
    """
    print(f"[ClimateIE] Loading spaCy sentence splitter ({spacy_model})")
    nlp = spacy.load(spacy_model, disable=["ner"])
 
    try:
        file_names = [f for f in os.listdir(data_dir) if f.endswith(".json")]
    except FileNotFoundError:
        print(f"[ClimateIE] ERROR: directory not found: {data_dir}")
        return []
 
    structured = []
    for file_name in file_names:
        file_path = os.path.join(data_dir, file_name)
        with open(file_path, "r", encoding="utf-8") as f:
            document = json.load(f)
 
        raw_text = document["text"]
        raw_entities = document.get("entities", {})
 
        nlp.max_length = len(raw_text) + 100_000
        doc = nlp(raw_text)
 
        entity_list = list(raw_entities.values())
        entity_list.sort(key=lambda x: x["begin"])
 
        for sent in doc.sents:
            sent_text = sent.text
            sent_start, sent_end = sent.start_char, sent.end_char
 
            local_entities = []
            for ent in entity_list:
                # Keep only entities strictly contained within this sentence
                if ent["begin"] >= sent_start and ent["end"] <= sent_end:
                    local_entities.append({
                        "text": ent["substring"],
                        "label": ent["label"],
                        "start": ent["begin"] - sent_start,
                        "end": ent["end"] - sent_start,
                    })
 
            if sent_text.strip():
                structured.append({
                    "text": sent_text,
                    "entities": local_entities,
                })
 
    print(f"[ClimateIE] {len(structured)} sentences from {len(file_names)} documents")
    return structured
